# Lab 3: Robot Localization with Kalman Filtering

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatayer8891-boop/zuj-robotics/blob/main/labs/lab-03-localization/notebook.ipynb)

---

**Course:** Introduction to Robotics for AI and Data Science (0135343)  
**Instructor:** Dr. Akram Fatayer · [a.fatayer@zuj.edu.jo](mailto:a.fatayer@zuj.edu.jo) · [LinkedIn](https://www.linkedin.com/in/akram-fatayer/)  
**University:** Al-Zaytoonah University of Jordan

---

## Overview

Implement a Kalman filter to fuse odometry and GPS data for accurate robot state estimation.

> **80/20 Principle:** Focus on understanding the core algorithm in each activity. The implementation details will follow naturally once you grasp the concept.


In [ ]:
# ============================================================
# COLAB ENVIRONMENT SETUP
# ============================================================
# This cell installs the required packages for running this lab
# in Google Colab. If you're running locally, you can skip this.
# ============================================================

import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Setting up Colab environment...")
    !pip install pybullet numpy matplotlib opencv-python-headless -q
    print("✅ All packages installed!")
    print("⚠️  Note: PyBullet runs in HEADLESS mode on Colab (no 3D GUI).")
    print("   You'll see matplotlib plots instead of the live 3D window.")
    print("   For the full 3D experience, run locally with: conda activate robotics_YourName")
else:
    print("✅ Running locally — full GUI mode available.")


---

## Activity 1: Activity1 Straight 1D

**File:** `activity1_straight_1d.py`

Run the code below to complete this activity.


In [ ]:

"""
Activity 1 — Straight-Line Motion (1‑D) with Constant‑Velocity Kalman Filter
Teaching version (well‑commented) — clean plots, no extra plot text boxes.

Learning goals (for students):
1) Understand predict → update cycle in a 1‑D setting.
2) See why a motion model (constant‑velocity) enables **prediction** between measurements.
3) Read uncertainty as a ±2σ (standard deviation) **band** around the estimate.

Key ideas in code:
- State x = [p, v]ᵀ (position and velocity along a line).
- Motion model (F) assumes constant velocity over each dt.
- Process noise (Q) comes from small random accelerations (white acceleration model).
- Measurement model (H) directly observes position (p) with noise variance R.
"""
import numpy as np
import matplotlib.pyplot as plt

# ---------------- Simulation ----------------
def simulate_straight_1d(num_steps=60, v=1.0, dt=1.0,
                          process_noise_std=0.05, meas_noise_std=1.0):
    """Generate 1‑D straight‑line motion with noisy position measurements.
    True motion includes a small perturbation so students see Q in action.
    """
    p = 0.0
    true_p = np.zeros(num_steps)
    z = np.zeros(num_steps)
    for k in range(num_steps):
        # True position update: p <- p + v*dt + process noise (small)
        p = p + v*dt + np.random.normal(0.0, process_noise_std)
        true_p[k] = p
        # Noisy position measurement
        z[k] = p + np.random.normal(0.0, meas_noise_std)
    return true_p, z

# ---------------- KF (CV model) ----------------
def run_kf_cv_1d(measurements, dt=1.0, p0=0.0, v0=0.0,
                 P0_pos_var=25.0, P0_vel_var=25.0,
                 accel_noise_std=0.1, meas_noise_std=1.0):
    """Run a 1‑D constant‑velocity KF on a sequence of position measurements.

    Parameters
    ----------
    measurements : array-like
        Noisy position z_k.
    dt : float
        Sampling period.
    p0, v0 : float
        Initial guesses for position and velocity.
    P0_pos_var, P0_vel_var : float
        Initial covariance (variance) for p and v (start large to allow learning).
    accel_noise_std : float
        Std of white acceleration driving Q (higher → more agile predictions).
    meas_noise_std : float
        Std of measurement noise (sets R = σ_z^2).
    """
    n = len(measurements)
    # State and covariance init
    x = np.array([p0, v0], dtype=float)
    P = np.diag([P0_pos_var, P0_vel_var])

    # Models: CV dynamics and position-only measurement
    F = np.array([[1.0, dt],
                  [0.0, 1.0]], dtype=float)
    H = np.array([1.0, 0.0], dtype=float)  # observe position only

    # Process noise from white acceleration (constant acceleration over dt approx.)
    s2 = accel_noise_std**2
    Q = s2 * np.array([[dt**4/4, dt**3/2],
                       [dt**3/2, dt**2  ]], dtype=float)
    R = meas_noise_std**2

    est = np.zeros(n)   # estimated positions
    var = np.zeros(n)   # position variances P[0,0]

    for k, zk in enumerate(measurements):
        # ---- Predict ----
        x = F @ x
        P = F @ P @ F.T + Q

        # ---- Update ---- (scalar measurement)
        y = zk - H @ x                 # innovation (residual)
        S = H @ P @ H.T + R            # innovation variance (scalar)
        K = P @ H / S                  # Kalman gain (2x1)
        x = x + K * y                  # state correction
        P = (np.eye(2) - np.outer(K, H)) @ P  # Joseph form not needed here

        est[k] = x[0]
        var[k] = P[0, 0]

    return est, var

# ---------------- Plotting ----------------
def plot_activity1(true_p, z, est, var, out_prefix='activity1'):
    """Make two clean plots: trajectory with ±2σ band, and error vs time.
    Keep visuals minimal (legend + grid) for a first exposure.
    """
    t = np.arange(len(true_p))
    std = np.sqrt(var)

    # Plot 1: Trajectory with ±2σ band
    plt.figure(figsize=(12, 5))
    plt.plot(t, true_p, 'g-', lw=2, label='True Position')
    plt.scatter(t, z, c='r', s=18, alpha=0.5, label='Noisy Measurements')
    plt.plot(t, est, 'b-', lw=2, label='KF Estimate')
    plt.fill_between(t, est - 2*std, est + 2*std, color='blue', alpha=0.2, label='±2σ band')
    plt.xlabel('Time step'); plt.ylabel('Position (m)')
    plt.title('Activity 1: 1‑D CV Kalman Filter — Straight Line')
    plt.grid(True, alpha=0.3); plt.legend()
    plt.tight_layout(); plt.savefig(out_prefix + '_traj.png', dpi=300)
    print('Saved', out_prefix + '_traj.png')

    # Plot 2: Estimation error (for intuition about bias and noise)
    err = est - true_p
    plt.figure(figsize=(12, 4))
    plt.plot(t, err, 'k-')
    plt.xlabel('Time step'); plt.ylabel('Position Error (m)')
    plt.title('Activity 1: Estimation Error over Time')
    plt.grid(True, alpha=0.3); plt.tight_layout(); plt.savefig(out_prefix + '_error.png', dpi=300)
    print('Saved', out_prefix + '_error.png')

# ---------------- Main ----------------

def main():
    # Deterministic seed so everyone sees the same figures in class.
    np.random.seed(42)

    # Simulation configuration (safe defaults for teaching)
    num_steps = 60
    dt = 1.0
    process_noise_std = 0.05
    measurement_noise_std = 1.0

    # Generate data
    true_p, z = simulate_straight_1d(num_steps=num_steps, v=1.0, dt=dt,
                                      process_noise_std=process_noise_std,
                                      meas_noise_std=measurement_noise_std)

    # Run KF
    est, var = run_kf_cv_1d(z, dt=dt, p0=0.0, v0=0.0,
                             P0_pos_var=25.0, P0_vel_var=25.0,
                             accel_noise_std=0.1,
                             meas_noise_std=measurement_noise_std)

    # Visualize
    plot_activity1(true_p, z, est, var)

if __name__ == '__main__':
    main()


---

## Activity 2: Activity2 Pybullet 2D V3

**File:** `activity2_pybullet_2d_v3.py`

Run the code below to complete this activity.


In [ ]:

"""
Activity 2 — 2D Kalman Filter with PyBullet Simulation (Instructor Version)
Version 3.0

Instructor goals:
1) Make the prediction step visible (prior vs posterior).
2) Keep the robot motion slow enough to watch in PyBullet.
3) Use baseline parameters that clearly show the role of Q and R.
4) Provide numerical metrics (RMSE) so students can verify performance.

State model:
    x = [x, y, vx, vy]^T
Measurement:
    z = [x, y]^T  (simulated noisy GPS)
"""

import time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

import pybullet as p
import pybullet_data
import sys
IN_COLAB = 'google.colab' in sys.modules




# ============================================================
# Helper functions
# ============================================================
def add_ellipse(ax, Pxy, mean_xy, n_std=2.0,
                facecolor='purple', edgecolor='purple',
                alpha=0.16, lw=1.2):
    """Draw a 2D covariance ellipse from a 2x2 covariance matrix."""
    vals, vecs = np.linalg.eigh(Pxy)
    order = np.argsort(vals)[::-1]
    vals = vals[order]
    vecs = vecs[:, order]

    lam1 = max(float(vals[0]), 0.0)
    lam2 = max(float(vals[1]), 0.0)

    width = 2.0 * n_std * np.sqrt(lam1)
    height = 2.0 * n_std * np.sqrt(lam2)
    angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))

    e = Ellipse(
        xy=mean_xy,
        width=width,
        height=height,
        angle=angle,
        facecolor=facecolor,
        edgecolor=edgecolor,
        alpha=alpha,
        lw=lw,
    )
    ax.add_patch(e)
    return e


def compute_rmse(est, truth):
    """Root Mean Squared Euclidean position error."""
    return np.sqrt(np.mean(np.sum((est - truth) ** 2, axis=1)))


# ============================================================
# PyBullet simulation
# ============================================================
def run_pybullet_simulation(
    num_steps=60,
    dt=0.2,
    gps_noise_std_true=0.35,
    use_gui=True,
    speed=1.0,
    start_height=0.5,
    camera_distance=6.0,
    camera_yaw=45,
    camera_pitch=-55,
):
    """
    Run a simple L-shaped trajectory in PyBullet using R2D2.

    Returns:
        true_pos: (N, 2) true x-y positions
        gps_pos:  (N, 2) noisy GPS-like measurements
        times:    (N,) time stamps
    """
    mode = p.GUI if use_gui else p.DIRECT
    client = p.connect(mode)

    p.setAdditionalSearchPath(pybullet_data.getDataPath())
    p.setGravity(0, 0, -10)

    # Use a small physics step for smooth motion.
    physics_dt = 1.0 / 240.0
    p.setTimeStep(physics_dt)
    substeps = max(1, int(round(dt / physics_dt)))

    # Optional: make the GUI camera friendly for class demos.
    if use_gui:
        p.resetDebugVisualizerCamera(
            cameraDistance=camera_distance,
            cameraYaw=camera_yaw,
            cameraPitch=camera_pitch,
            cameraTargetPosition=[2.0, 2.0, 0.0],
        )

    p.loadURDF("plane.urdf")
    robot_id = p.loadURDF(
        "r2d2.urdf",
        [0.0, 0.0, start_height],
        p.getQuaternionFromEuler([0.0, 0.0, 0.0]),
    )

    true_pos = np.zeros((num_steps, 2))
    gps_pos = np.zeros((num_steps, 2))
    times = np.arange(num_steps) * dt

    # L-shape: first half along +x, second half along +y
    for k in range(num_steps):
        if k < num_steps // 2:
            cmd_v = [speed, 0.0, 0.0]
        else:
            cmd_v = [0.0, speed, 0.0]

        p.resetBaseVelocity(robot_id, linearVelocity=cmd_v, angularVelocity=[0, 0, 0])

        for _ in range(substeps):
            p.stepSimulation()
            if use_gui:
                time.sleep(physics_dt)

        pos, _ = p.getBasePositionAndOrientation(robot_id)
        true_xy = np.array([pos[0], pos[1]])
        noise = np.random.normal(0.0, gps_noise_std_true, size=2)
        meas_xy = true_xy + noise

        true_pos[k] = true_xy
        gps_pos[k] = meas_xy

    p.disconnect(client)
    return true_pos, gps_pos, times


# ============================================================
# Kalman filter
# ============================================================
def run_kf_2d(
    measurements,
    dt=0.2,
    accel_noise_std=0.8,
    gps_noise_std_filter=0.8,
    dropout_range=None,
):
    """
    2D Constant-Velocity Kalman Filter.

    State:
        x = [x, y, vx, vy]^T

    This instructor version stores BOTH:
    - prediction (prior): x_{k|k-1}
    - corrected estimate (posterior): x_{k|k}

    Returns a dictionary with states, covariances, innovations, and gains.
    """
    n = len(measurements)

    # Initialize from first measurement for a cleaner classroom demonstration.
    x = np.array([measurements[0, 0], measurements[0, 1], 0.0, 0.0], dtype=float)
    P = np.diag([gps_noise_std_filter**2, gps_noise_std_filter**2, 4.0, 4.0])

    F = np.array([
        [1.0, 0.0, dt,  0.0],
        [0.0, 1.0, 0.0, dt ],
        [0.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 0.0, 1.0],
    ])

    H = np.array([
        [1.0, 0.0, 0.0, 0.0],
        [0.0, 1.0, 0.0, 0.0],
    ])

    # White-acceleration process noise for CV model.
    s2 = accel_noise_std ** 2
    Q = s2 * np.array([
        [dt**4 / 4.0, 0.0,          dt**3 / 2.0, 0.0],
        [0.0,          dt**4 / 4.0, 0.0,         dt**3 / 2.0],
        [dt**3 / 2.0,  0.0,         dt**2,       0.0],
        [0.0,          dt**3 / 2.0, 0.0,         dt**2],
    ])

    R = np.eye(2) * (gps_noise_std_filter ** 2)
    I = np.eye(4)

    pred_state = np.zeros((n, 4))
    pred_pos = np.zeros((n, 2))
    pred_cov_xy = []

    upd_state = np.zeros((n, 4))
    upd_pos = np.zeros((n, 2))
    upd_cov_xy = []

    innovations = np.full((n, 2), np.nan)
    gains = np.zeros((n, 4, 2))
    used_measurement = np.ones(n, dtype=bool)

    for k in range(n):
        # ---------------- Predict ----------------
        x_pred = F @ x
        P_pred = F @ P @ F.T + Q

        pred_state[k] = x_pred
        pred_pos[k] = x_pred[:2]
        pred_cov_xy.append(P_pred[:2, :2].copy())

        # Optional dropout window
        z = measurements[k].copy()
        if dropout_range is not None:
            start, end = dropout_range
            if start <= k < end:
                z[:] = np.nan
                used_measurement[k] = False

        # ---------------- Update ----------------
        if np.all(np.isfinite(z)):
            y = z - H @ x_pred                 # innovation / residual
            S = H @ P_pred @ H.T + R
            K = P_pred @ H.T @ np.linalg.inv(S)

            x = x_pred + K @ y

            # Joseph-form covariance update (numerically safer)
            IKH = I - K @ H
            P = IKH @ P_pred @ IKH.T + K @ R @ K.T

            innovations[k] = y
            gains[k] = K
        else:
            # No measurement available => posterior equals prior
            x = x_pred
            P = P_pred
            innovations[k] = np.array([np.nan, np.nan])
            gains[k] = 0.0

        upd_state[k] = x
        upd_pos[k] = x[:2]
        upd_cov_xy.append(P[:2, :2].copy())

    return {
        "pred_state": pred_state,
        "pred_pos": pred_pos,
        "pred_cov_xy": pred_cov_xy,
        "upd_state": upd_state,
        "upd_pos": upd_pos,
        "upd_cov_xy": upd_cov_xy,
        "innovations": innovations,
        "gains": gains,
        "used_measurement": used_measurement,
        "F": F,
        "H": H,
        "Q": Q,
        "R": R,
    }


# ============================================================
# Plotting
# ============================================================
def plot_tracking_results(
    true_pos,
    gps_pos,
    pred_pos,
    upd_pos,
    upd_cov_xy,
    used_measurement,
    ellipse_stride=5,
    save_path="activity2_pybullet_kf_v3.png",
):
    """Main trajectory plot showing truth, measurement, prediction, and update."""
    fig, ax = plt.subplots(figsize=(10, 8))

    ax.plot(true_pos[:, 0], true_pos[:, 1], 'g-', lw=2.5, label='True Path (PyBullet)')
    ax.scatter(gps_pos[:, 0], gps_pos[:, 1], c='r', s=25, alpha=0.5, label='Noisy GPS')
    ax.plot(pred_pos[:, 0], pred_pos[:, 1], color='orange', ls='--', lw=2.0, label='KF Prediction (Prior)')
    ax.plot(upd_pos[:, 0], upd_pos[:, 1], 'b-', lw=2.5, label='KF Update (Posterior)')

    # Draw small correction segments so students can SEE update = correction of prediction.
    for k in range(0, len(upd_pos), 2):
        ax.plot(
            [pred_pos[k, 0], upd_pos[k, 0]],
            [pred_pos[k, 1], upd_pos[k, 1]],
            color='gray', alpha=0.35, lw=1.0,
        )

    # Show covariance ellipses only on posterior for visual clarity.
    for k in range(0, len(upd_pos), ellipse_stride):
        color = 'purple' if used_measurement[k] else 'gray'
        add_ellipse(
            ax,
            upd_cov_xy[k],
            upd_pos[k],
            n_std=2.0,
            facecolor=color,
            edgecolor=color,
            alpha=0.14,
            lw=1.2,
        )

    ax.set_xlabel('X Position (m)')
    ax.set_ylabel('Y Position (m)')
    ax.set_title('2D Kalman Filter in PyBullet: Truth vs Measurement vs Prediction vs Update')
    ax.legend(loc='best')
    ax.grid(True)
    ax.axis('equal')

    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    print(f'Saved {save_path}')
    return fig, ax


def plot_innovations(times, innovations, save_path="activity2_innovations_v3.png"):
    """Plot measurement innovations (residuals)."""
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(times, innovations[:, 0], label='Innovation in x')
    ax.plot(times, innovations[:, 1], label='Innovation in y')
    ax.axhline(0.0, color='k', lw=1.0)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Residual')
    ax.set_title('Kalman Filter Innovation (Measurement Residual)')
    ax.grid(True)
    ax.legend(loc='best')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    print(f'Saved {save_path}')
    return fig, ax


def plot_position_errors(times, true_pos, gps_pos, pred_pos, upd_pos,
                         save_path="activity2_position_error_v3.png"):
    """Plot Euclidean position errors over time."""
    gps_err = np.linalg.norm(gps_pos - true_pos, axis=1)
    pred_err = np.linalg.norm(pred_pos - true_pos, axis=1)
    upd_err = np.linalg.norm(upd_pos - true_pos, axis=1)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(times, gps_err, 'r-', alpha=0.7, label='GPS error')
    ax.plot(times, pred_err, color='orange', ls='--', lw=2.0, label='Prediction error')
    ax.plot(times, upd_err, 'b-', lw=2.0, label='Updated KF error')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Euclidean position error (m)')
    ax.set_title('Position Error Over Time')
    ax.grid(True)
    ax.legend(loc='best')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    print(f'Saved {save_path}')
    return fig, ax


# ============================================================
# Main demo
# ============================================================
def main():
    np.random.seed(42)  # reproducible measurement noise

    # --------------------------------------------------------
    # Baseline parameters chosen for teaching clarity
    # --------------------------------------------------------
    num_steps = 60
    dt = 0.2
    speed = 1.0

    gps_noise_std_true = 0.35      # actual GPS noise in simulation
    gps_noise_std_filter = 0.35    # R used by the filter
    accel_noise_std = 0.8          # Q tuning parameter

    use_gui = True                 # set False for fast batch runs

    # Optional sensor dropout: e.g., (20, 28) or None
    dropout_range = None

    print('Running PyBullet simulation...')
    true_pos, gps_pos, times = run_pybullet_simulation(
        num_steps=num_steps,
        dt=dt,
        gps_noise_std_true=gps_noise_std_true,
        use_gui=use_gui,
        speed=speed,
    )

    print('Running Kalman filter...')
    kf = run_kf_2d(
        gps_pos,
        dt=dt,
        accel_noise_std=accel_noise_std,
        gps_noise_std_filter=gps_noise_std_filter,
        dropout_range=dropout_range,
    )

    pred_pos = kf['pred_pos']
    upd_pos = kf['upd_pos']
    upd_cov_xy = kf['upd_cov_xy']
    innovations = kf['innovations']
    used_measurement = kf['used_measurement']

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------
    gps_rmse = compute_rmse(gps_pos, true_pos)
    pred_rmse = compute_rmse(pred_pos, true_pos)
    upd_rmse = compute_rmse(upd_pos, true_pos)

    print('\n========== RMSE Summary =========')
    print(f'GPS RMSE           : {gps_rmse:.3f} m')
    print(f'Prediction RMSE    : {pred_rmse:.3f} m')
    print(f'Updated KF RMSE    : {upd_rmse:.3f} m')
    print('=================================\n')

    print('Filter matrices used:')
    print('F =\n', kf['F'])
    print('Q =\n', kf['Q'])
    print('R =\n', kf['R'])

    print('Plotting results...')
    plot_tracking_results(
        true_pos=true_pos,
        gps_pos=gps_pos,
        pred_pos=pred_pos,
        upd_pos=upd_pos,
        upd_cov_xy=upd_cov_xy,
        used_measurement=used_measurement,
        ellipse_stride=5,
        save_path='activity2_pybullet_kf_v3.png',
    )

    plot_innovations(
        times=times,
        innovations=innovations,
        save_path='activity2_innovations_v3.png',
    )

    plot_position_errors(
        times=times,
        true_pos=true_pos,
        gps_pos=gps_pos,
        pred_pos=pred_pos,
        upd_pos=upd_pos,
        save_path='activity2_position_error_v3.png',
    )

    plt.show()


if __name__ == '__main__':
    main()


---

## Activity 3: Activity3 Pybullet Fusion

**File:** `activity3_pybullet_fusion.py`

Run the code below to complete this activity.


In [ ]:
"""
Activity 3 — Sensor Fusion (Odometry + GPS) with PyBullet
Version 2.0 - Integrated with the Autonomous Car Project

=== Mission Context ===
Your R2D2 robot has TWO sensors, but neither is perfect:
  - Odometry (wheel encoders): Fast updates (10Hz), but drifts over time.
  - GPS: Drift-free, but noisy and slow (1Hz — only every 10 steps).

Your mission: Fuse both sensors using a Kalman Filter to get the BEST estimate.

Learning goals:
1) Understand the complementary nature of sensors (Odometry drifts, GPS is noisy).
2) Implement a Kalman Filter that fuses two different measurement sources.
3) See how uncertainty grows between GPS updates and shrinks when GPS arrives.

Key ideas:
- The robot moves in a circle in PyBullet.
- Odometry has a small constant bias → causes drift over time.
- GPS arrives only every 10 steps → sparse but drift-free.
- The KF combines both: smooth trajectory from odometry + corrections from GPS.
"""
import pybullet as p
import pybullet_data
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import sys
IN_COLAB = 'google.colab' in sys.modules



# ============================================================
# HELPER: Draw a 2D covariance ellipse on a matplotlib axis
# ============================================================
def add_ellipse(ax, Pxy, mean_xy, n_std=2.0, facecolor='purple',
                edgecolor='purple', alpha=0.15, lw=1.0):
    """
    Draws a 2D covariance ellipse representing ±n_std standard deviations.
    Pxy: 2x2 covariance matrix (position only).
    mean_xy: [x, y] center of the ellipse.
    """
    vals, vecs = np.linalg.eigh(Pxy)
    order = np.argsort(vals)[::-1]
    vals = vals[order]; vecs = vecs[:, order]
    a = 2.0 * n_std * np.sqrt(max(float(vals[0]), 0.0))
    b = 2.0 * n_std * np.sqrt(max(float(vals[1]), 0.0))
    theta_deg = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    e = Ellipse(xy=mean_xy, width=a, height=b, angle=theta_deg,
                facecolor=facecolor, edgecolor=edgecolor, alpha=alpha, lw=lw)
    ax.add_patch(e)

# ============================================================
# STEP 1: PyBullet Simulation — Robot moves in a circle
# ============================================================
def run_pybullet_simulation(num_steps=120, dt=0.1, odom_bias=0.05,
                            gps_noise_std=3.0, gps_freq=10):
    """
    Runs R2D2 in a circle using PyBullet.
    
    Returns:
        true_pos:  (N, 2) ground truth positions from PyBullet
        odom_vel:  (N, 2) noisy odometry velocity readings (with bias)
        gps_pos:   (N, 2) noisy GPS position readings (NaN when unavailable)
        gps_avail: (N,)   boolean array — True when GPS reading is available
    """
    # --- Connect to PyBullet (headless mode for data collection) ---
    physicsClient = p.connect(p.DIRECT)
    p.setAdditionalSearchPath(pybullet_data.getDataPath())
    p.setGravity(0, 0, -10)
    
    # --- Load environment and robot ---
    p.loadURDF("plane.urdf")
    
    # Circle parameters
    radius = 5.0
    omega = 2 * np.pi / (num_steps * dt)  # Angular velocity for one full circle
    
    # Start at the right edge of the circle (radius, 0)
    startPos = [radius, 0, 0.5]
    startOri = p.getQuaternionFromEuler([0, 0, 0])
    robotId = p.loadURDF("r2d2.urdf", startPos, startOri)
    
    # --- Allocate storage ---
    true_pos  = np.zeros((num_steps, 2))
    odom_vel  = np.zeros((num_steps, 2))
    gps_pos   = np.full((num_steps, 2), np.nan)
    gps_avail = np.zeros(num_steps, dtype=bool)
    
    # --- Simulation loop ---
    for i in range(num_steps):
        # Compute commanded velocity for circular motion
        angle = omega * i * dt
        vx_cmd = -omega * radius * np.sin(angle)
        vy_cmd =  omega * radius * np.cos(angle)
        
        # Apply velocity to robot
        p.resetBaseVelocity(robotId, linearVelocity=[vx_cmd, vy_cmd, 0],
                            angularVelocity=[0, 0, 0])
        p.stepSimulation()
        
        # --- Ground Truth (what PyBullet knows) ---
        pos, _ = p.getBasePositionAndOrientation(robotId)
        true_pos[i] = [pos[0], pos[1]]
        
        # --- Simulated Odometry Sensor ---
        # WHY bias? Real wheel encoders accumulate small systematic errors
        # (e.g., slightly different wheel diameters). This causes DRIFT.
        odom_noise = np.random.normal(0, 0.05, 2)
        odom_vel[i] = [vx_cmd + odom_bias + odom_noise[0],
                       vy_cmd + odom_bias + odom_noise[1]]
        
        # --- Simulated GPS Sensor ---
        # WHY sparse? Real GPS updates at ~1Hz, while odometry runs at 10-100Hz.
        # WHY noisy? GPS has ~3m accuracy in consumer-grade receivers.
        if i % gps_freq == 0:
            gps_noise = np.random.normal(0, gps_noise_std, 2)
            gps_pos[i] = [pos[0] + gps_noise[0], pos[1] + gps_noise[1]]
            gps_avail[i] = True
            
    p.disconnect()
    return true_pos, odom_vel, gps_pos, gps_avail

# ============================================================
# STEP 2: Sensor Fusion Kalman Filter
# ============================================================
def run_fusion_kf(odom_vel, gps_pos, gps_avail, dt=0.1,
                  accel_noise_std=5.0, odom_noise_std=0.05, gps_noise_std=3.0):
    """
    Fuses Odometry (velocity) and GPS (position) using a Kalman Filter.
    
    State vector: x = [px, py, vx, vy]
    Odometry measures: [vx, vy] (fast, biased)
    GPS measures: [px, py] (slow, noisy, drift-free)
    """
    n = len(odom_vel)
    
    # --- Initial state and covariance ---
    # We start at (5, 0) — the edge of the circle
    x = np.array([5.0, 0.0, 0.0, 0.0])
    P = np.eye(4) * 10.0  # High initial uncertainty (we're not sure)
    
    # --- State Transition Matrix (Constant Velocity model) ---
    # x_new = x_old + vx * dt
    # y_new = y_old + vy * dt
    F = np.array([
        [1, 0, dt, 0],
        [0, 1, 0, dt],
        [0, 0, 1,  0],
        [0, 0, 0,  1]
    ])
    
    # --- Process Noise Q ---
    # WHY large Q? The robot is moving in a CIRCLE, but our model assumes
    # STRAIGHT lines (constant velocity). Large Q tells the filter:
    # "My model is imperfect — trust measurements more."
    s2 = accel_noise_std**2
    Q = s2 * np.array([
        [dt**4/4, 0,       dt**3/2, 0      ],
        [0,       dt**4/4, 0,       dt**3/2],
        [dt**3/2, 0,       dt**2,   0      ],
        [0,       dt**3/2, 0,       dt**2  ]
    ])
    # Extra position noise to help track curves
    Q[0, 0] += 0.1
    Q[1, 1] += 0.1
    
    # --- Odometry Measurement Model ---
    # Odometry observes velocity: z_odom = [vx, vy]
    H_odom = np.array([[0, 0, 1, 0],
                       [0, 0, 0, 1]])
    R_odom = np.eye(2) * (odom_noise_std**2)
    
    # --- GPS Measurement Model ---
    # GPS observes position: z_gps = [px, py]
    H_gps = np.array([[1, 0, 0, 0],
                      [0, 1, 0, 0]])
    R_gps = np.eye(2) * (gps_noise_std**2)
    
    # --- Storage ---
    est_pos = np.zeros((n, 2))
    covariances = []
    
    for i in range(n):
        # ======== PREDICT ========
        x = F @ x
        P = F @ P @ F.T + Q
        
        # ======== UPDATE 1: Odometry (always available, every step) ========
        z_odom = odom_vel[i]
        y_odom = z_odom - H_odom @ x                    # Innovation
        S_odom = H_odom @ P @ H_odom.T + R_odom          # Innovation covariance
        K_odom = P @ H_odom.T @ np.linalg.inv(S_odom)    # Kalman Gain
        x = x + K_odom @ y_odom                           # Correct state
        P = (np.eye(4) - K_odom @ H_odom) @ P             # Correct covariance
        
        # ======== UPDATE 2: GPS (only when available) ========
        if gps_avail[i]:
            z_gps = gps_pos[i]
            y_gps = z_gps - H_gps @ x
            S_gps = H_gps @ P @ H_gps.T + R_gps
            K_gps = P @ H_gps.T @ np.linalg.inv(S_gps)
            x = x + K_gps @ y_gps
            P = (np.eye(4) - K_gps @ H_gps) @ P
            
        est_pos[i] = [x[0], x[1]]
        covariances.append(P[0:2, 0:2].copy())
        
    return est_pos, covariances

# ============================================================
# STEP 3: Baseline Methods (for comparison)
# ============================================================
def dead_reckoning(odom_vel, start_pos, dt=0.1):
    """
    Dead reckoning: integrates velocity to get position.
    This WILL drift because odometry has a bias.
    """
    n = len(odom_vel)
    pos = np.zeros((n, 2))
    pos[0] = start_pos
    for i in range(1, n):
        pos[i] = pos[i-1] + odom_vel[i] * dt
    return pos

def gps_hold(gps_pos, gps_avail, start_pos):
    """
    Holds the last known GPS position until a new one arrives.
    This is what you'd get if you ONLY had GPS — jumpy and sparse.
    """
    n = len(gps_pos)
    pos = np.zeros((n, 2))
    last = np.array(start_pos)
    for i in range(n):
        if gps_avail[i]:
            last = gps_pos[i].copy()
        pos[i] = last
    return pos

# ============================================================
# STEP 4: Visualization
# ============================================================
def plot_fusion_results(true_pos, odom_only, gps_only, fused_pos,
                        covariances, gps_avail):
    """Creates a 2x2 figure comparing all three approaches + uncertainty."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    # --- A. Odometry Only (Dead Reckoning) ---
    ax = axes[0, 0]
    ax.plot(true_pos[:, 0], true_pos[:, 1], 'g-', lw=3, label='True Path', zorder=3)
    ax.plot(odom_only[:, 0], odom_only[:, 1], 'b--', lw=2, label='Odometry (Drift)', zorder=2)
    ax.plot(true_pos[0, 0], true_pos[0, 1], 'go', ms=10, zorder=4)
    ax.set_title('A. Odometry Only (Dead Reckoning)', fontsize=13, fontweight='bold')
    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
    ax.legend(loc='upper left'); ax.grid(True, alpha=0.3); ax.axis('equal')
    
    # --- B. GPS Only ---
    ax = axes[0, 1]
    ax.plot(true_pos[:, 0], true_pos[:, 1], 'g-', lw=3, label='True Path', zorder=3)
    idx = np.where(gps_avail)[0]
    ax.scatter(gps_only[idx, 0], gps_only[idx, 1], c='r', marker='x', s=100,
               zorder=4, label='GPS Fixes')
    ax.plot(gps_only[:, 0], gps_only[:, 1], 'r-', lw=1, alpha=0.4, label='GPS Hold')
    ax.plot(true_pos[0, 0], true_pos[0, 1], 'go', ms=10, zorder=4)
    ax.set_title('B. GPS Only (Noisy & Sparse)', fontsize=13, fontweight='bold')
    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
    ax.legend(loc='upper left'); ax.grid(True, alpha=0.3); ax.axis('equal')
    
    # --- C. Sensor Fusion (The Winner!) ---
    ax = axes[1, 0]
    ax.plot(true_pos[:, 0], true_pos[:, 1], 'g-', lw=3, label='True Path', zorder=3)
    ax.plot(fused_pos[:, 0], fused_pos[:, 1], color='purple', lw=2,
            label='Fused (KF)', zorder=2)
    # Draw uncertainty ellipses every 10 steps (less cluttered)
    for i in range(0, len(fused_pos), 10):
        add_ellipse(ax, covariances[i], fused_pos[i], n_std=2.0)
    ax.plot(true_pos[0, 0], true_pos[0, 1], 'go', ms=10, zorder=4)
    ax.set_title('C. Sensor Fusion (Odom + GPS)', fontsize=13, fontweight='bold')
    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
    ax.legend(loc='upper left'); ax.grid(True, alpha=0.3); ax.axis('equal')
    
    # --- D. Uncertainty Evolution (Sawtooth Pattern) ---
    ax = axes[1, 1]
    t = np.arange(len(covariances))
    trP = np.array([np.trace(C) for C in covariances])
    ax.plot(t, trP, 'k-', lw=2, label='Uncertainty (Trace P)')
    ax.scatter(idx, trP[idx], c='red', marker='v', s=80, zorder=3, label='GPS Update')
    ax.set_title('D. Uncertainty Evolution (Sawtooth Pattern)', fontsize=13, fontweight='bold')
    ax.set_xlabel('Time Step'); ax.set_ylabel('Position Uncertainty (m²)')
    ax.legend(); ax.grid(True, alpha=0.3)
    
    plt.suptitle('Activity 3: Sensor Fusion — Odometry + GPS on PyBullet Robot',
                 fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('activity3_pybullet_fusion.png', dpi=300, bbox_inches='tight')
    print("Saved activity3_pybullet_fusion.png")

# ============================================================
# MAIN
# ============================================================
def main():
    np.random.seed(42)  # Reproducible results for all students
    
    # --- Configuration ---
    dt = 0.1
    gps_noise_std = 3.0   # GPS is noisy (~3m accuracy)
    odom_bias = 0.05      # Small constant bias in odometry
    start_pos = [5.0, 0.0]
    
    # --- Run Simulation ---
    print("Running PyBullet simulation (R2D2 moving in a circle)...")
    true_pos, odom_vel, gps_pos, gps_avail = run_pybullet_simulation(
        num_steps=120, dt=dt, odom_bias=odom_bias, gps_noise_std=gps_noise_std,
        gps_freq=10)
    
    # --- Compute Baselines ---
    print("Computing baselines (Dead Reckoning and GPS-only)...")
    odom_only = dead_reckoning(odom_vel, start_pos, dt=dt)
    gps_only  = gps_hold(gps_pos, gps_avail, start_pos)
    
    # --- Run Sensor Fusion KF ---
    print("Running Sensor Fusion Kalman Filter...")
    fused_pos, covariances = run_fusion_kf(
        odom_vel, gps_pos, gps_avail, dt=dt,
        accel_noise_std=5.0, odom_noise_std=0.05, gps_noise_std=gps_noise_std)
    
    # --- Visualize ---
    print("Generating comparison plots...")
    plot_fusion_results(true_pos, odom_only, gps_only, fused_pos,
                        covariances, gps_avail)
    
    # --- RMSE Comparison ---
    rmse_odom = np.sqrt(np.mean(np.sum((odom_only - true_pos)**2, axis=1)))
    rmse_gps  = np.sqrt(np.mean(np.sum((gps_only  - true_pos)**2, axis=1)))
    rmse_kf   = np.sqrt(np.mean(np.sum((fused_pos - true_pos)**2, axis=1)))
    
    print("\n" + "="*50)
    print("  RMSE COMPARISON")
    print("="*50)
    print(f"  Odometry only (Dead Reckoning): {rmse_odom:.3f} m  ← Worst (drift)")
    print(f"  GPS only (Hold):                {rmse_gps:.3f} m  ← Noisy")
    print(f"  Kalman Filter (Fused):          {rmse_kf:.3f} m  ← BEST!")
    print("="*50)
    print(f"\n  Fusion improved over GPS by: {(1 - rmse_kf/rmse_gps)*100:.1f}%")
    print(f"  Fusion improved over Odom by: {(1 - rmse_kf/rmse_odom)*100:.1f}%")

if __name__ == '__main__':
    main()


---

## Summary & Next Steps

Congratulations on completing this lab! Before moving on:

1. **Review** your outputs and make sure they match expected behavior
2. **Experiment** by modifying parameters and observing the effects
3. **Reflect** on what you learned — write a brief paragraph in your report

---

*Dr. Akram Fatayer · [a.fatayer@zuj.edu.jo](mailto:a.fatayer@zuj.edu.jo) · [LinkedIn](https://www.linkedin.com/in/akram-fatayer/) · Al-Zaytoonah University of Jordan*
